# Inspect a synthetic prior batch

Load one on-disk prior batch (512 datasets by default) from
`examples/run_genload_continuous.sh` or `examples/run_genload_discrete.sh`.

Set `PRIOR_DIR` below if needed.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lifelines import CoxPHFitter, KaplanMeierFitter
from pycox.evaluation import EvalSurv
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from sic.prior.genload import sparse2dense

REPO_ROOT = Path("..").resolve() if (Path.cwd().name == "examples") else Path(".").resolve()
PRIOR_DIR = REPO_ROOT / "data" / "prior_continuous"
BATCH_IDX = 0
RANDOM_SEED = 0
TEST_SIZE = 0.3
COX_PENALIZER = 0.1

In [ ]:
def load_prior_batch(prior_dir: Path, batch_idx: int = 0):
    prior_dir = Path(prior_dir)
    batch_path = prior_dir / "batch_data" / f"batch_{batch_idx:06d}.pt"
    if not batch_path.exists():
        raise FileNotFoundError(f"{batch_path} not found — run a genload script first.")

    batch = torch.load(batch_path, map_location="cpu", weights_only=True)
    y = batch["y"]
    d = batch["d"]
    batch_size = int(batch["batch_size"])

    X = batch["X"]
    if X.dim() == 1:
        seq_lens = batch["seq_lens"]
        T = int(seq_lens[0].item())
        max_features = int(d.max().item())
        row_lengths = d.repeat_interleave(T)
        X = sparse2dense(X, row_lengths, max_len=max_features).view(batch_size, T, max_features)

    times = y[:, 0, :].numpy()
    events = y[:, 1, :].numpy().astype(bool)
    return X.numpy(), times, events, d.numpy()


X, times, events, n_features = load_prior_batch(PRIOR_DIR, BATCH_IDX)
n_datasets = times.shape[0]

## One random synthetic dataset

In [ ]:
rng = np.random.default_rng()
ds_idx = int(rng.integers(n_datasets))
t, e = times[ds_idx], events[ds_idx]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

kmf = KaplanMeierFitter()
kmf.fit(t, e)
kmf.plot_survival_function(ax=axes[0], ci_show=True)
if axes[0].get_legend():
    axes[0].get_legend().remove()
axes[0].set_title(f"KM curve (dataset {ds_idx})")
axes[0].set_xlabel("time")

axes[1].hist(t[e], bins=40, alpha=0.7, label="events")
axes[1].hist(t[~e], bins=40, alpha=0.7, label="censored")
axes[1].set_title(f"Observed times (dataset {ds_idx})")
axes[1].set_xlabel("time")
axes[1].legend()

plt.tight_layout()
plt.show()

## All 512 datasets on a standardized time scale

Per dataset, times are scaled to \([0, 1]\) via \(t / \max(t)\). Left: KM curves. Right: pooled observed times (events vs censored).

In [ ]:
def standardize_times(t: np.ndarray) -> np.ndarray:
    t_max = t.max()
    return t / t_max if t_max > 0 else t


fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for i in range(n_datasets):
    t_std = standardize_times(times[i])
    kmf = KaplanMeierFitter()
    kmf.fit(t_std, events[i])
    kmf.plot_survival_function(ax=axes[0], ci_show=False, alpha=0.08, linewidth=0.8, color="C0")

if axes[0].get_legend():
    axes[0].get_legend().remove()
axes[0].set_title(f"KM curves — all {n_datasets} datasets")
axes[0].set_xlabel("standardized time")
axes[0].set_ylabel("S(t)")

t_std_events = np.concatenate([standardize_times(times[i])[events[i]] for i in range(n_datasets)])
t_std_censored = np.concatenate([standardize_times(times[i])[~events[i]] for i in range(n_datasets)])

axes[1].hist(t_std_events, bins=60, density=True, alpha=0.7, label="events")
axes[1].hist(t_std_censored, bins=60, density=True, alpha=0.7, label="censored")
axes[1].set_title(f"Observed times — all {n_datasets} datasets")
axes[1].set_xlabel("standardized time")
axes[1].legend()

plt.tight_layout()
plt.show()

## Cox PH baseline: td-C-index per dataset

Use each synthetic cohort in full, split with `train_test_split`, fit `lifelines.CoxPHFitter`, and report td-C-index on the held-out subjects.

In [ ]:
def coxph_td_cindex(X_i, t, e, n_feat, random_state):
    n_feat = int(n_feat)
    cols = [f"x{j}" for j in range(n_feat)]
    df = pd.DataFrame(X_i[:, :n_feat], columns=cols)
    df["T"] = t
    df["E"] = e.astype(int)

    stratify = df["E"] if df["E"].nunique() > 1 else None
    df_tr, df_te = train_test_split(
        df, test_size=TEST_SIZE, random_state=random_state, stratify=stratify,
    )

    if df_tr["E"].sum() < 2 or df_te["E"].sum() < 1 or len(df_te) < 2:
        return np.nan

    cph = CoxPHFitter(penalizer=COX_PENALIZER)
    cph.fit(df_tr, duration_col="T", event_col="E")

    surv = cph.predict_survival_function(df_te[cols])
    t_te = df_te["T"].to_numpy()
    e_te = df_te["E"].to_numpy()

    return EvalSurv(surv, t_te, e_te, censor_surv="km").concordance_td()


td_cindices = []
for i in tqdm(range(n_datasets), desc="Cox PH"):
    try:
        ci = coxph_td_cindex(X[i], times[i], events[i], n_features[i], random_state=RANDOM_SEED + i)
    except Exception:
        ci = np.nan
    td_cindices.append(ci)

td_cindices = np.array(td_cindices)
valid = td_cindices[~np.isnan(td_cindices)]
print(f"td-C-index: {valid.mean():.3f} ± {valid.std():.3f}  (n={len(valid)}/{n_datasets})")
print(f"datasets with td-C-index > 0.5: {(valid > 0.5).sum()} / {len(valid)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0.5, 1.0, 50)
ax.hist(valid, bins=bins, edgecolor="white")
ax.axvline(0.5, color="C3", linestyle="--", linewidth=1)
ax.set_xlabel("td-C-index")
ax.set_ylabel("number of datasets")
ax.set_title(f"Cox PH td-C-index across {len(valid)} datasets")
plt.tight_layout()
plt.show()